# BCI-FedAdapt — Ablation Study

Runs **all 7 FL strategies** on **BCI-IV 2a** then **BCI-IV 2b** to provide a
rigorous ablation of BCI-FedAdapt against standard federated learning methods.

| Strategy | Key mechanism |
|----------|--------------|
| HFL-FedAvg | Weighted average — baseline |
| HFL-FedProx | FedAvg + proximal penalty μ=0.02 |
| HFL-SCAFFOLD | Control-variate variance reduction |
| HFL-Momentum | EMA of aggregated weights β=0.9 |
| HFL-FedNova | Normalised averaging (Δw / steps) |
| HFL-FedBS | FedAvg + class-balanced sampling |
| **BCI-FedAdapt** | Quality-weighted + layer-selective + Phase 2 personalisation |

### Fair comparison note
BCI-FedAdapt is compared at Phase 2 (personalised) accuracy. All baselines use
a single global model. To validate that the gain comes from the algorithm (not
just extra local compute), run `HFL-FedAvg + 20 local fine-tuning epochs` as an
additional ablation baseline.

In [ ]:
# 1. Install
import subprocess, sys
def pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(a))
pip('moabb', 'mne', 'scikit-learn', 'torch', 'tqdm', 'matplotlib', 'seaborn', 'scipy')
print('Install complete')

In [ ]:
# 2. Imports
import warnings, copy, random, os
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from typing import Dict, List, Tuple, Optional
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR
from scipy.signal import butter, filtfilt, iirnotch
import moabb
from moabb.datasets import BNCI2014_001, BNCI2014_004
from moabb.paradigms import MotorImagery
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
moabb.set_log_level('WARNING')

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs('/mnt/user-data/outputs', exist_ok=True)
print(f'Device: {DEVICE}')

In [ ]:
# Config — BCI-IV 2a
CFG_2A = dict(
    dataset_name = 'BCI-IV 2a',
    sfreq=250, tmin=0.0, tmax=4.0, fmin=4.0, fmax=40.0,
    n_classes=4, n_channels=22,
    notch_freq=50.0, apply_car=True, apply_ea=True, ea_eps=1e-6, norm_epochs=True,
    window_size=500, step_size=125,
    F1=16, D=2, F2=32, dropout=0.4,
    n_heads=4, attn_dp=0.4, tcn_filters=32, tcn_kernels=4, tcn_dp=0.3,
    local_epochs=10, batch_size=64, lr=3e-3, wd=1e-4,
    grad_clip=1.0, label_smoothing=0.05, eta_min=1e-5,
    aug_prob=0.5, aug_noise_std=0.01,
    num_rounds=30, proximal_mu=0.02, scaffold_lr=3e-3, momentum_beta=0.9,
    # BCI-FedAdapt
    qa_alpha=0.4, qa_beta=0.4, qa_gamma=0.2, qa_temperature=0.5,
    personal_epochs=20, personal_lr=5e-4, personal_wd=1e-5,
)
print('CFG_2A ready')

# Config — BCI-IV 2b   *** FIX: added personal_epochs + qa_* keys ***
CFG_2B = dict(
    dataset_name='BCI-IV 2b',
    sfreq=250, tmin=0.0, tmax=4.0, fmin=8.0, fmax=30.0,
    n_classes=2, n_channels=3,
    apply_car=True, apply_ea=True, ea_eps=1e-6, norm_epochs=True,
    window_size=500, step_size=125,
    F1=8, D=2, F2=16, dropout=0.35,
    n_heads=2, attn_dp=0.3, tcn_filters=16, tcn_kernels=4, tcn_dp=0.2,
    local_epochs=10, batch_size=64, lr=3e-3, wd=1e-4,
    grad_clip=1.0, label_smoothing=0.1, eta_min=1e-5,
    aug_prob=0.5, aug_noise_std=0.01,
    num_rounds=30, proximal_mu=0.02, scaffold_lr=3e-3, momentum_beta=0.9,
    # BCI-FedAdapt  ← these were missing (root cause of KeyError)
    qa_alpha=0.4, qa_beta=0.4, qa_gamma=0.2, qa_temperature=0.5,
    personal_epochs=20, personal_lr=5e-4, personal_wd=1e-5,
)
print('CFG_2B ready')

STRATEGIES = [
    'HFL-FedAvg', 'HFL-FedProx', 'HFL-SCAFFOLD',
    'HFL-Momentum', 'HFL-FedNova', 'HFL-FedBS', 'BCI-FedAdapt'
]
print('Strategies:', STRATEGIES)

## Preprocessing + Data Loading

In [ ]:
# Preprocessing helpers
def apply_car(X: np.ndarray) -> np.ndarray:
    return X - X.mean(axis=1, keepdims=True)

def euclidean_alignment(X: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    N, C, T = X.shape
    R = np.zeros((C, C), dtype=np.float64)
    for i in range(N):
        xi = X[i].astype(np.float64)
        R += xi @ xi.T / T
    R /= N
    R += eps * np.eye(C)
    eigvals, eigvecs = np.linalg.eigh(R)
    eigvals = np.maximum(eigvals, eps)
    R_inv_sqrt = eigvecs @ np.diag(1.0 / np.sqrt(eigvals)) @ eigvecs.T
    return np.einsum('ij,njt->nit', R_inv_sqrt, X).astype(np.float32)

def normalize_epochs(X: np.ndarray) -> np.ndarray:
    mu  = X.mean(axis=2, keepdims=True)
    std = X.std(axis=2, keepdims=True) + 1e-8
    return (X - mu) / std

def preprocess(X: np.ndarray, cfg: dict) -> np.ndarray:
    if cfg.get('apply_car', True):  X = apply_car(X)
    if cfg.get('apply_ea',  True):  X = euclidean_alignment(X, eps=cfg.get('ea_eps', 1e-6))
    if cfg.get('norm_epochs', True): X = normalize_epochs(X)
    return X

def make_windows(X: np.ndarray, y: np.ndarray, window_size: int, step_size: int):
    N, C, T = X.shape
    wins, labels = [], []
    for i in range(N):
        for start in range(0, T - window_size + 1, step_size):
            wins.append(X[i, :, start:start + window_size])
            labels.append(y[i])
    return np.array(wins, dtype=np.float32), np.array(labels, dtype=np.int64)

def soft_vote_predict(model: nn.Module, X_trials: np.ndarray,
                      window_size: int, step_size: int) -> np.ndarray:
    model.eval()
    preds = []
    with torch.no_grad():
        for trial in X_trials:
            C, T = trial.shape
            windows = []
            for s in range(0, T - window_size + 1, step_size):
                windows.append(trial[:, s:s + window_size])
            wins_t = torch.tensor(np.array(windows), dtype=torch.float32).to(DEVICE)
            probs  = F.softmax(model(wins_t), dim=1).mean(0)
            preds.append(probs.argmax().item())
    return np.array(preds)

print('Preprocessing + window helpers defined')

In [ ]:
# Load and preprocess dataset
def load_and_preprocess(cfg: dict) -> Tuple[dict, int, object]:
    name = cfg.get('dataset_name', 'BCI-IV 2a')
    print(f'Loading {name} ...')
    if '2a' in name or '001' in name:
        dataset = BNCI2014_001()
    else:
        dataset = BNCI2014_004()
    paradigm = MotorImagery(
        n_classes=cfg['n_classes'], fmin=cfg['fmin'], fmax=cfg['fmax'],
        tmin=cfg['tmin'], tmax=cfg['tmax'], resample=cfg['sfreq'])
    X, y, meta = paradigm.get_data(dataset=dataset)
    le    = LabelEncoder()
    y_enc = le.fit_transform(y)
    print(f'  Raw shape: {X.shape}  Classes: {list(le.classes_)}')
    cdata = {}
    for cid, subj in enumerate(sorted(meta['subject'].unique())):
        mask  = (meta['subject'] == subj).values
        Xs, ys = preprocess(X[mask].astype(np.float32), cfg), y_enc[mask].astype(np.int64)
        Xtr, Xte, ytr, yte = train_test_split(
            Xs, ys, test_size=0.2, stratify=ys, random_state=SEED)
        cdata[cid] = dict(X_train=Xtr, X_test=Xte, y_train=ytr, y_test=yte)
        print(f'  Client {cid:02d}: train={len(ytr)}  test={len(yte)}')
    n_times = X.shape[-1]
    return cdata, n_times, le.classes_

print('load_and_preprocess defined')

## ATCNet Architecture

In [ ]:
# ATCNet architecture
class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=4, dilation=1, dp=0.3):
        super().__init__()
        self.ks, self.dil = kernel_size, dilation
        self.conv1    = nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation)
        self.conv2    = nn.Conv1d(out_ch, out_ch, kernel_size, dilation=dilation)
        self.bn1      = nn.BatchNorm1d(out_ch); self.bn2 = nn.BatchNorm1d(out_ch)
        self.act      = nn.ELU(); self.drop = nn.Dropout(dp)
        self.residual = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def _pad(self, x):
        total = self.dil * (self.ks - 1)
        return F.pad(x, (total // 2, total - total // 2))
    def forward(self, x):
        T = x.size(2); res = self.residual(x)
        h = self.drop(self.act(self.bn1(self.conv1(self._pad(x))[:, :, :T])))
        h = self.drop(self.act(self.bn2(self.conv2(self._pad(h))[:, :, :T])))
        return h + res

class ATCNet(nn.Module):
    def __init__(self, n_classes, n_channels, n_times, F1=16, D=2, F2=32, dp=0.4,
                 n_heads=4, attn_dp=0.4, tcn_filters=32, tcn_kernels=4, tcn_dp=0.3):
        super().__init__()
        self.temporal  = nn.Sequential(
            nn.Conv2d(1, F1, (1, 64), padding=(0, 32), bias=False), nn.BatchNorm2d(F1))
        self.spatial   = nn.Sequential(
            nn.Conv2d(F1, D*F1, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(D*F1), nn.ELU(), nn.AvgPool2d((1, 4)), nn.Dropout(dp))
        self.separable = nn.Sequential(
            nn.Conv2d(D*F1, D*F1, (1, 16), padding=(0, 8), groups=D*F1, bias=False),
            nn.Conv2d(D*F1, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2), nn.ELU(), nn.AvgPool2d((1, 8)), nn.Dropout(dp))
        with torch.no_grad():
            _x = torch.zeros(1, 1, n_channels, n_times)
            self.T_feat = self.separable(self.spatial(self.temporal(_x))).shape[-1]
        self.attn_norm = nn.LayerNorm(F2)
        self.mha       = nn.MultiheadAttention(F2, n_heads, dropout=attn_dp, batch_first=True)
        self.attn_drop = nn.Dropout(attn_dp)
        self.tcn1      = TCNBlock(F2, tcn_filters, tcn_kernels, dilation=1, dp=tcn_dp)
        self.tcn2      = TCNBlock(tcn_filters, tcn_filters, tcn_kernels, dilation=2, dp=tcn_dp)
        self.out_norm  = nn.LayerNorm(tcn_filters)
        self.fc        = nn.Linear(tcn_filters, n_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.separable(self.spatial(self.temporal(x)))
        x = x.squeeze(2).transpose(1, 2)
        res = x
        x, _ = self.mha(self.attn_norm(x), self.attn_norm(x), self.attn_norm(x))
        x = self.attn_drop(x) + res
        x = x.transpose(1, 2)
        x = self.tcn2(self.tcn1(x))
        return self.fc(self.out_norm(x.mean(dim=2)))

def make_model(cfg: dict) -> ATCNet:
    return ATCNet(cfg['n_classes'], cfg['n_channels'], cfg['window_size'],
                  F1=cfg['F1'], D=cfg['D'], F2=cfg['F2'], dp=cfg['dropout'],
                  n_heads=cfg['n_heads'], attn_dp=cfg['attn_dp'],
                  tcn_filters=cfg['tcn_filters'], tcn_kernels=cfg['tcn_kernels'],
                  tcn_dp=cfg['tcn_dp']).to(DEVICE)

_m = make_model(CFG_2A)
_x = torch.randn(4, CFG_2A['n_channels'], CFG_2A['window_size']).to(DEVICE)
print(f'ATCNet output: {_m(_x).shape}')
print(f'Params       : {sum(p.numel() for p in _m.parameters()):,}')

## FL Strategy Implementations

In [ ]:
# HFL aggregation helpers + HFLClient + run_hfl  (all non-FedAdapt strategies)
def weighted_avg(weight_list, ns):
    total = sum(ns)
    agg   = [torch.zeros_like(w) for w in weight_list[0]]
    for ws, n in zip(weight_list, ns):
        a = n / total
        for acc, w in zip(agg, ws): acc.add_(w * a)
    return agg

def aggregate(clients, strategy, prev_agg=None, c_global=None, cfg={}):
    weights = [c.get_weights() for c in clients]
    ns      = [c.n_train for c in clients]
    if strategy in ('HFL-FedAvg', 'HFL-FedProx', 'HFL-FedBS'):
        return weighted_avg(weights, ns), c_global
    elif strategy == 'HFL-SCAFFOLD':
        agg = weighted_avg(weights, ns)
        if c_global is None: c_global = [torch.zeros_like(p) for p in agg]
        new_cg = [torch.zeros_like(c) for c in c_global]
        for client in clients:
            for nc, ci in zip(new_cg, client.c_i): nc.add_(ci / len(clients))
        return agg, new_cg
    elif strategy == 'HFL-Momentum':
        beta = cfg.get('momentum_beta', 0.9)
        agg  = weighted_avg(weights, ns)
        if prev_agg is None: prev_agg = [w.clone() for w in agg]
        mom = [beta * pa + (1 - beta) * a for pa, a in zip(prev_agg, agg)]
        return mom, mom
    elif strategy == 'HFL-FedNova':
        agg_base  = weighted_avg(weights, ns)
        total     = sum(ns)
        nova      = [[(w - b) / max(c._nova_steps, 1) for w, b in zip(c.get_weights(), agg_base)]
                     for c in clients]
        delta     = weighted_avg(nova, ns)
        avg_steps = sum(c._nova_steps * c.n_train for c in clients) / total
        return [b + avg_steps * d for b, d in zip(agg_base, delta)], c_global
    raise ValueError(f'Unknown strategy: {strategy}')

class HFLClient:
    def __init__(self, client_id, data, cfg):
        self.cid = client_id; self.cfg = cfg; self.n_train = len(data['y_train'])
        ws, ss = cfg['window_size'], cfg['step_size']
        X_win, y_win = make_windows(data['X_train'], data['y_train'], ws, ss)
        Xt, yt = torch.from_numpy(X_win), torch.from_numpy(y_win)
        self.train_loader = DataLoader(TensorDataset(Xt, yt),
                                       batch_size=cfg['batch_size'], shuffle=True, drop_last=True)
        counts  = np.bincount(y_win); weights = 1.0 / counts[y_win]
        sampler = WeightedRandomSampler(torch.from_numpy(weights).float(), len(weights))
        self.balanced_loader = DataLoader(TensorDataset(Xt, yt),
                                          batch_size=cfg['batch_size'],
                                          sampler=sampler, drop_last=True)
        self.X_test_raw = data['X_test']; self.y_test = data['y_test']
        self.model      = make_model(cfg)
        self.criterion  = nn.CrossEntropyLoss(label_smoothing=cfg.get('label_smoothing', 0.0))
        self.c_i        = [torch.zeros_like(p) for p in self.model.parameters()]
        self._nova_steps = 0

    def get_weights(self): return [p.data.clone() for p in self.model.parameters()]
    def set_weights(self, ws):
        for p, w in zip(self.model.parameters(), ws): p.data.copy_(w)
    def evaluate(self):
        ws, ss = self.cfg['window_size'], self.cfg['step_size']
        preds  = soft_vote_predict(self.model, self.X_test_raw, ws, ss)
        return accuracy_score(self.y_test, preds), len(self.y_test)

    def local_train(self, strategy, w_global=None, c_global=None):
        self.model.train()
        opt   = torch.optim.Adam(self.model.parameters(),
                                 lr=self.cfg['lr'], weight_decay=self.cfg['wd'])
        sched = CosineAnnealingLR(opt, T_max=self.cfg['local_epochs'],
                                  eta_min=self.cfg['eta_min'])
        w0 = [p.data.clone() for p in self.model.parameters()]
        loader = self.balanced_loader if strategy == 'HFL-FedBS' else self.train_loader
        total_loss, steps = 0.0, 0
        for _ in range(self.cfg['local_epochs']):
            for Xb, yb in loader:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                if self.cfg.get('aug_prob', 0) > 0 and random.random() < self.cfg['aug_prob']:
                    Xb = Xb + torch.randn_like(Xb) * self.cfg.get('aug_noise_std', 0.01)
                opt.zero_grad(); logits = self.model(Xb); loss = self.criterion(logits, yb)
                if strategy == 'HFL-FedProx' and w_global is not None:
                    prox = sum((p - g.to(DEVICE)).pow(2).sum()
                               for p, g in zip(self.model.parameters(), w_global))
                    loss = loss + (self.cfg['proximal_mu'] / 2) * prox
                loss.backward()
                if strategy == 'HFL-SCAFFOLD' and c_global is not None:
                    for p, ci, cg in zip(self.model.parameters(), self.c_i, c_global):
                        if p.grad is not None: p.grad.data.add_(cg.to(DEVICE) - ci.to(DEVICE))
                nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg['grad_clip'])
                opt.step(); total_loss += loss.item(); steps += 1
            sched.step()
        if strategy == 'HFL-SCAFFOLD' and c_global is not None:
            lr_s = self.cfg['scaffold_lr']
            with torch.no_grad():
                for p, w_, ci, cg in zip(self.model.parameters(), w0, self.c_i, c_global):
                    ci.copy_(ci - cg.to(DEVICE) + (w_ - p.data) / (steps * lr_s + 1e-9))
        self._nova_steps = steps
        return total_loss / max(steps, 1)

def run_hfl(strategy, cfg, client_data):
    n_clients = len(client_data)
    print(f'\n  {strategy}  rounds={cfg["num_rounds"]}')
    init_w  = [p.data.clone() for p in make_model(cfg).parameters()]
    clients = [HFLClient(cid, client_data[cid], cfg) for cid in range(n_clients)]
    for c in clients: c.set_weights(init_w)
    agg_w, c_global, history = init_w, None, []
    for rnd in range(1, cfg['num_rounds'] + 1):
        losses = [c.local_train(strategy, w_global=agg_w, c_global=c_global) for c in clients]
        agg_w, c_global = aggregate(clients, strategy,
                                     prev_agg=agg_w if strategy == 'HFL-Momentum' else None,
                                     c_global=c_global, cfg=cfg)
        for c in clients: c.set_weights(agg_w)
        accs, ns  = zip(*[c.evaluate() for c in clients])
        mean_acc  = float(np.average(accs, weights=ns))
        history.append(mean_acc)
        if rnd % 5 == 0 or rnd == 1:
            print(f'  Round {rnd:3d}  loss={np.mean(losses):.4f}  acc={mean_acc:.4f}')
    return {'history': history, 'final_weights': agg_w}

print('HFL strategies + run_hfl defined')

In [ ]:
# BCI-FedAdapt core: layer taxonomy, quality scoring, client, aggregator, runner
UNIVERSAL_PREFIXES = ('temporal', 'separable', 'attn_norm', 'mha', 'tcn1', 'tcn2', 'out_norm')
PERSONAL_PREFIXES  = ('spatial', 'fc')

def is_universal(name): return name.startswith(UNIVERSAL_PREFIXES)
def is_personal(name):  return name.startswith(PERSONAL_PREFIXES)

def flatten_params(state_dict):
    parts = [v.reshape(-1).float() for v in state_dict.values()
             if v.dtype in (torch.float32, torch.float16, torch.bfloat16)]
    return torch.cat(parts) if parts else torch.zeros(1)

def compute_quality_weights(clients, w_univ_before, cfg):
    n = len(clients)
    cos_scores = np.zeros(n); val_scores = np.zeros(n); stab_scores = np.zeros(n)
    deltas = []
    for client in clients:
        w_after = {k: v for k, v in client.model.state_dict().items() if is_universal(k)}
        delta   = {k: w_after[k] - w_univ_before[k].to(DEVICE) for k in w_after}
        deltas.append(flatten_params(delta))
    mean_delta = torch.stack(deltas).mean(0)
    for i, (client, delta) in enumerate(zip(clients, deltas)):
        cos_scores[i] = float(F.cosine_similarity(
            delta.unsqueeze(0), mean_delta.unsqueeze(0)).clamp(-1, 1))
        acc, _ = client.evaluate()
        val_scores[i] = acc
        stab_scores[i] = float(np.var(client._grad_norms)) if len(client._grad_norms) > 1 else 0.0
    max_gnv   = stab_scores.max() + 1e-9
    stab_norm = 1.0 - stab_scores / max_gnv
    a, b, g   = cfg['qa_alpha'], cfg['qa_beta'], cfg['qa_gamma']
    scores    = a * cos_scores + b * val_scores + g * stab_norm
    tau       = cfg['qa_temperature']
    exp_s     = np.exp((scores - scores.max()) / tau)
    weights   = exp_s / exp_s.sum()
    return weights, dict(cos=cos_scores, val=val_scores, stab=stab_norm,
                         composite=scores, weights=weights)


class BCIFedAdaptClient:
    def __init__(self, client_id, data, cfg):
        self.cid  = client_id; self.cfg = cfg
        self.n_train = len(data['y_train'])
        ws, ss = cfg['window_size'], cfg['step_size']
        X_win, y_win = make_windows(data['X_train'], data['y_train'], ws, ss)
        Xt, yt = torch.from_numpy(X_win), torch.from_numpy(y_win)
        counts  = np.bincount(y_win)
        weights = 1.0 / counts[y_win]
        sampler = WeightedRandomSampler(torch.from_numpy(weights).float(), len(weights))
        self.balanced_loader = DataLoader(TensorDataset(Xt, yt),
                                          batch_size=cfg['batch_size'],
                                          sampler=sampler, drop_last=True)
        self.X_test_raw = data['X_test']; self.y_test = data['y_test']
        self.model     = make_model(cfg)
        self.criterion = nn.CrossEntropyLoss(label_smoothing=cfg.get('label_smoothing', 0.0))
        self._grad_norms: List[float] = []

    def get_universal_state(self):
        return {k: v.clone() for k, v in self.model.state_dict().items() if is_universal(k)}

    def set_universal_state(self, state):
        current = self.model.state_dict()
        current.update({k: v.to(DEVICE) for k, v in state.items()})
        self.model.load_state_dict(current)

    def evaluate(self):
        ws, ss = self.cfg['window_size'], self.cfg['step_size']
        preds  = soft_vote_predict(self.model, self.X_test_raw, ws, ss)
        return accuracy_score(self.y_test, preds), len(self.y_test)

    def local_train(self, strategy='BCI-FedAdapt'):
        self._grad_norms = []
        self.model.train()
        opt   = torch.optim.Adam(self.model.parameters(),
                                 lr=self.cfg['lr'], weight_decay=self.cfg['wd'])
        sched = CosineAnnealingLR(opt, T_max=self.cfg['local_epochs'],
                                  eta_min=self.cfg['eta_min'])
        total_loss, steps = 0.0, 0
        for _ in range(self.cfg['local_epochs']):
            for Xb, yb in self.balanced_loader:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                if random.random() < self.cfg.get('aug_prob', 0):
                    Xb = Xb + torch.randn_like(Xb) * self.cfg.get('aug_noise_std', 0.01)
                opt.zero_grad()
                loss = self.criterion(self.model(Xb), yb)
                loss.backward()
                gn = sum(p.grad.data.norm(2).item()**2 for p in self.model.parameters()
                         if p.grad is not None) ** 0.5
                self._grad_norms.append(gn)
                nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg['grad_clip'])
                opt.step()
                total_loss += loss.item(); steps += 1
            sched.step()
        return total_loss / max(steps, 1)

    def personalise(self):
        for name, param in self.model.named_parameters():
            param.requires_grad = is_personal(name)
        personal_params = [p for p in self.model.parameters() if p.requires_grad]
        if not personal_params:
            return self.evaluate()[0]
        opt   = torch.optim.Adam(personal_params,
                                 lr=self.cfg['personal_lr'], weight_decay=self.cfg['personal_wd'])
        sched = CosineAnnealingLR(opt, T_max=self.cfg['personal_epochs'],
                                  eta_min=self.cfg['eta_min'])
        self.model.train()
        for _ in range(self.cfg['personal_epochs']):
            for Xb, yb in self.balanced_loader:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                opt.zero_grad()
                loss = self.criterion(self.model(Xb), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(personal_params, self.cfg['grad_clip'])
                opt.step()
            sched.step()
        for p in self.model.parameters():
            p.requires_grad = True
        return self.evaluate()[0]


def aggregate_bci_fedadapt(clients, w_univ_snapshot, cfg, round_num):
    weights, diag = compute_quality_weights(clients, w_univ_snapshot, cfg)
    agg_state = {}
    for key in clients[0].get_universal_state().keys():
        agg_state[key] = torch.zeros_like(
            clients[0].get_universal_state()[key], dtype=torch.float32)
        for client, w in zip(clients, weights):
            agg_state[key] += w * client.get_universal_state()[key].float()
    for client in clients:
        client.set_universal_state(agg_state)
    return agg_state, weights, diag


def run_bci_fedadapt(cfg: dict, client_data: dict) -> dict:
    n_clients = len(client_data)
    print(f'\n{"-"*72}')
    print(f'  BCI-FedAdapt  |  Phase1: {cfg["num_rounds"]} rounds  Phase2: {cfg["personal_epochs"]} epochs')
    print(f'{"-"*72}')
    init_state = make_model(cfg).state_dict()
    clients = []
    for cid in range(n_clients):
        c = BCIFedAdaptClient(cid, client_data[cid], cfg)
        c.model.load_state_dict({k: v.clone().to(DEVICE) for k, v in init_state.items()})
        clients.append(c)
    agg_univ_state = {k: v.clone().to(DEVICE) for k, v in init_state.items() if is_universal(k)}
    history, weight_log, diag_log = [], [], []

    print('  [Phase 1] Federated training...')
    for rnd in range(1, cfg['num_rounds'] + 1):
        w_snap  = {k: v.clone() for k, v in agg_univ_state.items()}
        losses  = [c.local_train() for c in clients]
        agg_univ_state, q_weights, diag = aggregate_bci_fedadapt(clients, w_snap, cfg, rnd)
        weight_log.append(q_weights.copy()); diag_log.append(diag)
        accs, ns = zip(*[c.evaluate() for c in clients])
        mean_acc = float(np.average(accs, weights=ns))
        history.append(mean_acc)
        if rnd % 5 == 0 or rnd == 1:
            top = int(np.argmax(q_weights))
            print(f'  Round {rnd:3d}  loss={np.mean(losses):.4f}  acc={mean_acc:.4f}  '
                  f'top=C{top}({q_weights[top]:.3f})')

    print(f'\n  [Phase 2] Local personalisation ({cfg["personal_epochs"]} epochs)...')
    personal_accs_before = [c.evaluate()[0] for c in clients]
    personal_accs_after  = []
    for cid, client in enumerate(clients):
        acc_after = client.personalise()
        personal_accs_after.append(acc_after)
        print(f'  Client {cid:02d}: before={personal_accs_before[cid]:.4f}  '
              f'after={acc_after:.4f}  Δ={acc_after-personal_accs_before[cid]:+.4f}')

    accs2, ns2 = zip(*[c.evaluate() for c in clients])
    mean_acc2  = float(np.average(accs2, weights=ns2))
    print(f'\n  Phase1 final: {history[-1]:.4f}  Phase2 final: {mean_acc2:.4f}  '
          f'(Δ={mean_acc2-history[-1]:+.4f})')
    return {
        'history': history, 'weight_log': np.array(weight_log), 'diag_log': diag_log,
        'personal_accs_before': personal_accs_before, 'personal_accs_after': personal_accs_after,
        'mean_acc_phase1': history[-1], 'mean_acc_phase2': mean_acc2,
        'final_weights': [p.data.clone() for p in clients[0].model.parameters()],
        'final_personal_states': {cid: clients[cid].model.state_dict() for cid in range(n_clients)},
        'clients': clients,
    }

print('BCI-FedAdapt (layer taxonomy, client, aggregator, runner) defined')

In [ ]:
# Global evaluation helper
def eval_global(weights, cfg, client_data, personal_states=None):
    ws, ss = cfg['window_size'], cfg['step_size']
    all_preds, all_labels = [], []
    for cid in range(len(client_data)):
        model = make_model(cfg)
        if personal_states is not None:
            model.load_state_dict({k: v.clone().to(DEVICE)
                                   for k, v in personal_states[cid].items()})
        else:
            for p, w in zip(model.parameters(), weights): p.data.copy_(w)
        preds = soft_vote_predict(model, client_data[cid]['X_test'], ws, ss)
        all_preds.extend(preds)
        all_labels.extend(client_data[cid]['y_test'])
    yp = np.array(all_preds); yl = np.array(all_labels)
    return accuracy_score(yl, yp), yl, yp

print('eval_global defined')

In [ ]:
# All visualisation functions

def plot_convergence(results, cfg, tag, save, strategies=None):
    strats  = strategies or list(results.keys())
    palette = sns.color_palette('tab10', len(strats))
    rounds  = range(1, cfg['num_rounds'] + 1)
    chance  = 1.0 / cfg['n_classes']
    plt.figure(figsize=(11, 4))
    for i, s in enumerate(strats):
        lw = 3.0 if s == 'BCI-FedAdapt' else 1.5
        plt.plot(rounds, results[s]['history'], label=s, color=palette[i], lw=lw)
    plt.axhline(chance, ls='--', color='grey', lw=1, label=f'Chance ({chance:.2f})')
    plt.xlabel('Round'); plt.ylabel('Mean Client Test Accuracy')
    plt.title(f'FL Convergence — {tag}')
    plt.legend(loc='lower right', fontsize=9, ncol=2); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(save, dpi=150); plt.show()
    print(f'Saved: {save}')

def plot_quality_weights(results, cfg, tag, save):
    if 'BCI-FedAdapt' not in results: return
    r    = results['BCI-FedAdapt']
    wlog = r['weight_log']
    nr, nc = wlog.shape
    rounds = np.arange(1, nr + 1)
    pal    = sns.color_palette('tab10', nc)
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes[0].stackplot(rounds, wlog.T, labels=[f'C{i}' for i in range(nc)],
                      colors=pal, alpha=0.85)
    axes[0].set_xlabel('Round'); axes[0].set_ylabel('Weight (stacked)')
    axes[0].set_title(f'Aggregation Weight Share — {tag}')
    axes[0].set_xlim(1, nr); axes[0].legend(loc='upper right', fontsize=7, ncol=3)
    sns.heatmap(wlog.T, cmap='YlOrRd', vmin=0, vmax=wlog.max(), linewidths=0.2,
                xticklabels=[str(r) if r % 5 == 0 else '' for r in rounds],
                yticklabels=[f'C{i}' for i in range(nc)],
                cbar_kws={'label': 'Quality weight'}, ax=axes[1])
    axes[1].set_xlabel('Round'); axes[1].set_title(f'Quality Weight Heatmap — {tag}')
    plt.tight_layout(); plt.savefig(save, dpi=150); plt.show()

def plot_personalisation_gain(results, cfg, tag, save):
    if 'BCI-FedAdapt' not in results: return
    r      = results['BCI-FedAdapt']
    before = r['personal_accs_before']; after = r['personal_accs_after']
    deltas = [a - b for a, b in zip(after, before)]
    n      = len(before); pal = sns.color_palette('tab10', n)
    x      = np.arange(n)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(x-0.2, before, 0.38, label='Phase 1', color=[p+(0.6,) for p in pal], edgecolor='w')
    axes[0].bar(x+0.2, after,  0.38, label='Phase 2', color=pal, edgecolor='w')
    axes[0].set_xticks(x); axes[0].set_xticklabels([f'C{i}' for i in range(n)])
    axes[0].set_ylim(0, 1); axes[0].set_ylabel('Accuracy')
    axes[0].set_title(f'Phase 1 vs Phase 2 per Client — {tag}')
    axes[0].legend(); axes[0].grid(alpha=0.2, axis='y')
    axes[0].axhline(1/cfg['n_classes'], ls='--', color='grey', lw=1)
    cols = ['#27ae60' if d >= 0 else '#e74c3c' for d in deltas]
    axes[1].bar(x, [d*100 for d in deltas], color=cols, edgecolor='w')
    axes[1].axhline(0, color='grey', lw=0.8)
    axes[1].set_xticks(x); axes[1].set_xticklabels([f'C{i}' for i in range(n)])
    axes[1].set_ylabel('Gain (%)'); axes[1].set_title(f'Phase 2 Gain per Client — {tag}')
    for i, d in enumerate(deltas):
        axes[1].text(i, d*100+(0.3 if d >= 0 else -0.9), f'{d*100:+.1f}%', ha='center', fontsize=8)
    axes[1].grid(alpha=0.2, axis='y')
    plt.suptitle(f'BCI-FedAdapt — Two-Phase Personalisation [{tag}]', y=1.02)
    plt.tight_layout(); plt.savefig(save, dpi=150); plt.show()
    print(f'Phase1 mean: {np.mean(before):.4f}  Phase2 mean: {np.mean(after):.4f}  '
          f'Δ={np.mean(after)-np.mean(before):+.4f}')

def plot_quality_components(results, cfg, tag, save):
    if 'BCI-FedAdapt' not in results: return
    dlog   = results['BCI-FedAdapt']['diag_log']
    last_n = min(5, len(dlog))
    nc     = len(dlog[-1]['cos'])
    x      = np.arange(nc)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, (key, label, col) in zip(axes, [
            ('cos',  'Gradient Consensus (cos-sim)', '#3498db'),
            ('val',  'Local Val Accuracy',            '#e67e22'),
            ('stab', 'Gradient Stability (1-var)',    '#27ae60')]):
        for r in range(last_n):
            d = dlog[-(last_n-r)]
            ax.plot(x, d[key], alpha=0.4+r*0.12, color=col, lw=1.5, marker='o', ms=4,
                    label=f'Rnd {cfg["num_rounds"]-last_n+r+1}')
        ax.set_xticks(x); ax.set_xticklabels([f'C{i}' for i in range(nc)])
        ax.set_title(label); ax.set_xlabel('Client'); ax.grid(alpha=0.3); ax.legend(fontsize=7)
    plt.suptitle(f'Quality Score Components — Last 5 Rounds [{tag}]', y=1.04)
    plt.tight_layout(); plt.savefig(save, dpi=150); plt.show()

def plot_eval(final, strategies, cfg, class_names, tag, best, save_prefix):
    acc_b, yl, yp = final[best]
    cm      = confusion_matrix(yl, yp)
    palette = sns.color_palette('tab10', len(strategies))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, ax=axes[0])
    axes[0].set_title(f'Confusion Matrix — {best} [{tag}]  (acc={acc_b:.4f})')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
    strats = list(final.keys()); accs = [final[s][0] for s in strats]
    cols   = ['#e74c3c' if s == best else '#3498db' for s in strats]
    axes[1].barh(strats, accs, color=cols, edgecolor='white', height=0.6)
    axes[1].axvline(1.0/cfg['n_classes'], ls='--', color='grey', lw=1, label='Chance')
    axes[1].set_xlim(0, 1.05); axes[1].set_xlabel('Global Test Accuracy')
    axes[1].set_title(f'Final Global Accuracy — {tag}')
    for i, (s, a) in enumerate(zip(strats, accs)):
        axes[1].text(a+0.005, i, f'{a:.4f}', va='center', fontsize=9)
    axes[1].legend(); plt.tight_layout()
    plt.savefig(f'{save_prefix}_eval.png', dpi=150); plt.show()
    print(f'\nClassification Report — {best} [{tag}]')
    print(classification_report(yl, yp, target_names=class_names))

def plot_per_client_heatmap(results, strategies, cfg, client_data, tag, save):
    ws, ss = cfg['window_size'], cfg['step_size']
    per_client = {}
    for strat in strategies:
        accs = []
        for cid in range(len(client_data)):
            model = make_model(cfg)
            if strat == 'BCI-FedAdapt' and 'final_personal_states' in results[strat]:
                model.load_state_dict({k: v.clone().to(DEVICE)
                                       for k, v in results[strat]['final_personal_states'][cid].items()})
            else:
                for p, w in zip(model.parameters(), results[strat]['final_weights']): p.data.copy_(w)
            preds = soft_vote_predict(model, client_data[cid]['X_test'], ws, ss)
            accs.append(accuracy_score(client_data[cid]['y_test'], preds))
        per_client[strat] = accs
    df = pd.DataFrame(per_client, index=[f'Client {i}' for i in range(len(client_data))])
    plt.figure(figsize=(max(8, len(strategies)*1.7), len(client_data)*0.8+1))
    sns.heatmap(df, annot=True, fmt='.3f', cmap='YlOrRd',
                vmin=1/cfg['n_classes'], vmax=1.0, linewidths=0.5,
                cbar_kws={'label': 'Accuracy'})
    plt.title(f'Per-Client Test Accuracy — Final Model [{tag}]')
    plt.tight_layout(); plt.savefig(save, dpi=150); plt.show()

def plot_radar_comparison(final, strategies, cfg, client_data, results, tag, save):
    ws, ss = cfg['window_size'], cfg['step_size']
    metrics = {}
    for strat in strategies:
        per_c = []
        for cid in range(len(client_data)):
            model = make_model(cfg)
            if strat == 'BCI-FedAdapt' and 'final_personal_states' in results[strat]:
                model.load_state_dict({k: v.clone().to(DEVICE)
                                       for k, v in results[strat]['final_personal_states'][cid].items()})
            else:
                for p, w in zip(model.parameters(), results[strat]['final_weights']): p.data.copy_(w)
            preds = soft_vote_predict(model, client_data[cid]['X_test'], ws, ss)
            per_c.append(accuracy_score(client_data[cid]['y_test'], preds))
        chance = 1.0 / cfg['n_classes']
        metrics[strat] = {
            'Global Acc'  : final[strat][0],
            'Best Client' : max(per_c),
            'Worst Client': min(per_c),
            'Equitability': 1.0 - float(np.std(per_c)),
            'Gain/Chance' : min(final[strat][0] / chance, 3.0) / 3.0,
        }
    labels   = list(next(iter(metrics.values())).keys())
    n_labels = len(labels)
    angles   = np.linspace(0, 2*np.pi, n_labels, endpoint=False).tolist()
    angles  += angles[:1]
    fig, ax  = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    palette  = sns.color_palette('tab10', len(strategies))
    for i, strat in enumerate(strategies):
        values  = [metrics[strat][l] for l in labels] + [metrics[strat][labels[0]]]
        lw      = 3.0 if strat == 'BCI-FedAdapt' else 1.5
        ls      = '-'  if strat == 'BCI-FedAdapt' else '--'
        ax.plot(angles, values, color=palette[i], lw=lw, ls=ls, label=strat)
        ax.fill(angles, values, color=palette[i], alpha=0.06)
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels, size=10); ax.set_ylim(0, 1)
    ax.set_title(f'Strategy Comparison Radar — {tag}\n(BCI-FedAdapt = solid thick)', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=8)
    plt.tight_layout(); plt.savefig(save, dpi=150, bbox_inches='tight'); plt.show()

def print_summary(final, strategies, cfg, results, tag, best):
    SEP = '=' * 80
    desc = {
        'HFL-FedAvg'   : 'Weighted avg of client weights',
        'HFL-FedProx'  : f'FedAvg + proximal penalty μ={cfg.get("proximal_mu", 0.02)}',
        'HFL-SCAFFOLD' : 'Control-variate variance reduction',
        'HFL-Momentum' : f'Momentum aggregation β={cfg.get("momentum_beta", 0.9)}',
        'HFL-FedNova'  : 'Normalised averaging — Δw / local_steps',
        'HFL-FedBS'    : 'FedAvg + class-balanced WeightedRandomSampler',
        'BCI-FedAdapt' : 'Quality-weighted layer-selective + Phase 2 personalisation',
    }
    print(SEP)
    print(f'  {tag}  |  Clients: {len(final)}  Rounds: {cfg["num_rounds"]}  Device: {DEVICE}')
    print(SEP)
    print(f'  {"Strategy":<20}  {"Acc":>8}  {"BestRnd":>8}  Description')
    print(f'  {"-"*20}  {"-"*8}  {"-"*8}  {"-"*42}')
    for s in strategies:
        g_acc = final[s][0]; b_acc = max(results[s]['history'])
        flag  = '  ← BEST' if s == best else ''
        print(f'  {s:<20}  {g_acc:>8.4f}  {b_acc:>8.4f}  {desc.get(s, "")}{flag}')
    print(f'\n  Chance ({cfg["n_classes"]}-class): {1/cfg["n_classes"]:.4f}')
    print(SEP)

print('All plot/summary functions defined')

---
# Part A — BCI-IV 2a  (9 subjects, 4-class)

In [ ]:
client_data_2a, N_TIMES_2A, CLASSES_2A = load_and_preprocess(CFG_2A)
CFG_2A['n_times'] = N_TIMES_2A
CLASS_NAMES_2A    = [str(c) for c in CLASSES_2A]
print(f'Classes 2a: {CLASS_NAMES_2A}')

In [ ]:
# Run all strategies — BCI-IV 2a
results_2a = {}
for strat in STRATEGIES:
    if strat == 'BCI-FedAdapt':
        results_2a[strat] = run_bci_fedadapt(CFG_2A, client_data_2a)
    else:
        results_2a[strat] = run_hfl(strat, CFG_2A, client_data_2a)
print('\n[2a] All strategies complete.')

## Results — BCI-IV 2a

In [ ]:
# Convergence — all strategies 2a
plot_convergence(results_2a, CFG_2A, 'BCI-IV 2a',
                 '/mnt/user-data/outputs/abl_convergence_2a.png')

In [ ]:
# Final evaluation — BCI-IV 2a
final_2a = {}
for strat in STRATEGIES:
    ps  = results_2a[strat].get('final_personal_states') if strat == 'BCI-FedAdapt' else None
    acc, yl, yp = eval_global(results_2a[strat]['final_weights'],
                               CFG_2A, client_data_2a, personal_states=ps)
    final_2a[strat] = (acc, yl, yp)
    note = ' [personalised]' if strat == 'BCI-FedAdapt' else ''
    print(f'{strat:<20}  Acc = {acc:.4f}{note}')
best_2a = max(final_2a, key=lambda s: final_2a[s][0])
print(f'\nBest: {best_2a}  ({final_2a[best_2a][0]:.4f})')

In [ ]:
# Radar chart — all strategies 2a
plot_radar_comparison(final_2a, STRATEGIES, CFG_2A, client_data_2a, results_2a,
                      'BCI-IV 2a', '/mnt/user-data/outputs/abl_radar_2a.png')

In [ ]:
# Confusion matrix + accuracy bar — 2a
plot_eval(final_2a, STRATEGIES, CFG_2A, CLASS_NAMES_2A,
          'BCI-IV 2a', best_2a, '/mnt/user-data/outputs/abl_2a')

In [ ]:
# Per-client accuracy heatmap — 2a
plot_per_client_heatmap(results_2a, STRATEGIES, CFG_2A, client_data_2a,
                        'BCI-IV 2a', '/mnt/user-data/outputs/abl_per_client_2a.png')

In [ ]:
# Quality weights (BCI-FedAdapt only) — 2a
plot_quality_weights(results_2a, CFG_2A, 'BCI-IV 2a',
                     '/mnt/user-data/outputs/abl_weights_2a.png')

In [ ]:
# Personalisation gain (BCI-FedAdapt only) — 2a
plot_personalisation_gain(results_2a, CFG_2A, 'BCI-IV 2a',
                          '/mnt/user-data/outputs/abl_personalisation_2a.png')

In [ ]:
# Summary table — 2a
print_summary(final_2a, STRATEGIES, CFG_2A, results_2a, 'BCI-IV 2a', best_2a)

---
# Part B — BCI-IV 2b  (9 subjects, 2-class)

In [ ]:
client_data_2b, N_TIMES_2B, CLASSES_2B = load_and_preprocess(CFG_2B)
CFG_2B['n_times'] = N_TIMES_2B
CLASS_NAMES_2B    = [str(c) for c in CLASSES_2B]
print(f'Classes 2b: {CLASS_NAMES_2B}')

In [ ]:
# Run all strategies — BCI-IV 2b
results_2b = {}
for strat in STRATEGIES:
    if strat == 'BCI-FedAdapt':
        results_2b[strat] = run_bci_fedadapt(CFG_2B, client_data_2b)
    else:
        results_2b[strat] = run_hfl(strat, CFG_2B, client_data_2b)
print('\n[2b] All strategies complete.')

## Results — BCI-IV 2b

In [ ]:
plot_convergence(results_2b, CFG_2B, 'BCI-IV 2b',
                 '/mnt/user-data/outputs/abl_convergence_2b.png')

In [ ]:
# Final evaluation — BCI-IV 2b
final_2b = {}
for strat in STRATEGIES:
    ps  = results_2b[strat].get('final_personal_states') if strat == 'BCI-FedAdapt' else None
    acc, yl, yp = eval_global(results_2b[strat]['final_weights'],
                               CFG_2B, client_data_2b, personal_states=ps)
    final_2b[strat] = (acc, yl, yp)
    note = ' [personalised]' if strat == 'BCI-FedAdapt' else ''
    print(f'{strat:<20}  Acc = {acc:.4f}{note}')
best_2b = max(final_2b, key=lambda s: final_2b[s][0])
print(f'\nBest: {best_2b}  ({final_2b[best_2b][0]:.4f})')

In [ ]:
plot_radar_comparison(final_2b, STRATEGIES, CFG_2B, client_data_2b, results_2b,
                      'BCI-IV 2b', '/mnt/user-data/outputs/abl_radar_2b.png')

In [ ]:
plot_eval(final_2b, STRATEGIES, CFG_2B, CLASS_NAMES_2B,
          'BCI-IV 2b', best_2b, '/mnt/user-data/outputs/abl_2b')

In [ ]:
plot_per_client_heatmap(results_2b, STRATEGIES, CFG_2B, client_data_2b,
                        'BCI-IV 2b', '/mnt/user-data/outputs/abl_per_client_2b.png')

In [ ]:
plot_quality_weights(results_2b, CFG_2B, 'BCI-IV 2b',
                     '/mnt/user-data/outputs/abl_weights_2b.png')

In [ ]:
plot_personalisation_gain(results_2b, CFG_2B, 'BCI-IV 2b',
                          '/mnt/user-data/outputs/abl_personalisation_2b.png')

In [ ]:
print_summary(final_2b, STRATEGIES, CFG_2B, results_2b, 'BCI-IV 2b', best_2b)

---
# Cross-Dataset Comparison

In [ ]:
# Cross-dataset comparison — both datasets side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
palette   = sns.color_palette('tab10', len(STRATEGIES))
x         = np.arange(len(STRATEGIES))
w         = 0.35
accs_2a   = [final_2a[s][0] for s in STRATEGIES]
accs_2b   = [final_2b[s][0] for s in STRATEGIES]

bars1 = axes[0].bar(x-w/2, accs_2a, w, label='BCI-IV 2a (4-class)',
                    color=[p+(0.8,) for p in palette], edgecolor='white')
bars2 = axes[0].bar(x+w/2, accs_2b, w, label='BCI-IV 2b (2-class)',
                    color=palette, edgecolor='white')
axes[0].axhline(0.25, ls=':', color='#e74c3c', lw=1.2, label='2a chance')
axes[0].axhline(0.50, ls=':', color='#3498db', lw=1.2, label='2b chance')
axes[0].set_xticks(x)
axes[0].set_xticklabels([s.replace('HFL-','').replace('BCI-','BCI\n')
                          for s in STRATEGIES], rotation=20, ha='right', fontsize=8)
axes[0].set_ylabel('Global Test Accuracy')
axes[0].set_title('Final Accuracy — 2a vs 2b  (★ = BCI-FedAdapt)')
axes[0].set_ylim(0, 1.08); axes[0].legend(fontsize=8)
for bar, a2a in zip(bars1, accs_2a):
    axes[0].text(bar.get_x()+bar.get_width()/2, a2a+0.01,
                 f'{a2a:.3f}', ha='center', va='bottom', fontsize=6)
for bar, a2b in zip(bars2, accs_2b):
    axes[0].text(bar.get_x()+bar.get_width()/2, a2b+0.01,
                 f'{a2b:.3f}', ha='center', va='bottom', fontsize=6)
if 'BCI-FedAdapt' in STRATEGIES:
    fa_idx = STRATEGIES.index('BCI-FedAdapt')
    for xpos, a, c in [(fa_idx-w/2, accs_2a[fa_idx], palette[fa_idx]),
                        (fa_idx+w/2, accs_2b[fa_idx], palette[fa_idx])]:
        axes[0].scatter([xpos], [a+0.04], marker='*', s=120, color='gold',
                        zorder=5, edgecolors='black', lw=0.5)

rounds = np.arange(1, CFG_2A['num_rounds']+1)
for tag, final, results, cfg, col in [
        ('2a', final_2a, results_2a, CFG_2A, '#e74c3c'),
        ('2b', final_2b, results_2b, CFG_2B, '#3498db')]:
    best_s = max(final, key=lambda s: final[s][0])
    for s in STRATEGIES:
        lw = 2.0 if s == best_s else 0.6
        alpha = 0.7 if s == 'BCI-FedAdapt' else 0.25
        axes[1].plot(rounds, results[s]['history'], color=col, lw=lw, alpha=alpha,
                     label=f'{tag}: {s}' if s == best_s else None)
axes[1].axhline(0.25, ls=':', color='#e74c3c', lw=1); axes[1].axhline(0.50, ls=':', color='#3498db', lw=1)
axes[1].set_xlabel('Round'); axes[1].set_ylabel('Mean Client Accuracy')
axes[1].set_title('Convergence — best (bold) & all strategies (faint)')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/cross_dataset_comparison.png', dpi=150)
plt.show()

## Final Research Summary

In [ ]:
# Final research summary
SEP = '=' * 84
print(SEP)
print('  BCI-FedAdapt ABLATION STUDY — FULL RESULTS')
print(SEP)
for tag, final, results, cfg, best in [
        ('BCI-IV 2a', final_2a, results_2a, CFG_2A, best_2a),
        ('BCI-IV 2b', final_2b, results_2b, CFG_2B, best_2b)]:
    print(f'\n  {tag}:')
    fedadapt_acc = final.get('BCI-FedAdapt', (0,))[0]
    for s in STRATEGIES:
        delta = final[s][0] - fedadapt_acc if s != 'BCI-FedAdapt' else 0
        tag2  = '' if s == 'BCI-FedAdapt' else f'  ({delta:+.4f} vs BCI-FedAdapt)'
        marker = '  ← BEST' if s == best else ''
        print(f'    {s:<20}  {final[s][0]:.4f}{tag2}{marker}')
print(SEP)